In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
from analysis_framework import Dataset
from ReweightingHelper import ReweightingHelper
from AltSetupHandler import AltSetupHandler

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x4048cf0
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x94a8c90


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# prod = False
prod = True
no_rvec = True
write_outputs = False
# write_outputs = True
# dataset_path = "data/datasets/selected-objects/test.json"
# output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/oo-sqme/test"
# output_meta_path = "data/datasets/oo-sqme"
# output_meta = f"{output_meta_path}/test.json"
# checked_output_meta = f"{output_meta_path}/checked-test.json"
output_collections = r"(\w*sqme\w*)"
if prod:
    dataset_path = "data/datasets/selected-objects-new2/signal-only.json"
    output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/oo-sqme-new2/signal-only-cc10"
    output_meta_path = "data/datasets/oo-sqme-new2"
    output_meta = f"{output_meta_path}/signal-only-cc10.json"

In [4]:
# ROOT.EnableImplicitMT(n_threads)
environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ReweightingHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xcbd9600


In [7]:
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "mW": 80.419,
    "g1z": 1.0,
    "ka": 1.0,
    "la": 0.0,
    "mmu": 0.000511
  },
"variations": [
    1e-08
  ]
}
""", mirror=False, combinations=False)
alt_configs = alt_setup_handler.get_alt_setup()
print(alt_configs)
analysis.initialise_omega_wrappers(alt_configs, lib_path="OO/whizard/cc10_ac_inclusive/.libs/default_lib.so", nominal_pars={"mmu": 0.000511})

{'mW_pos_1em08': {'mW': 80.41900000999999, 'g1z': 1.0, 'ka': 1.0, 'la': 0.0, 'mmu': 0.000511}, 'g1z_pos_1em08': {'mW': 80.419, 'g1z': 1.00000001, 'ka': 1.0, 'la': 0.0, 'mmu': 0.000511}, 'ka_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.00000001, 'la': 0.0, 'mmu': 0.000511}, 'la_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.0, 'la': 1e-08, 'mmu': 0.000511}, 'mmu_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.0, 'la': 0.0, 'mmu': 0.00051101}}


cling::DynamicLibraryManager::loadLibrary(): libHepMC3rootIO.so.3: cannot open shared object file: No such file or directory


In [ ]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [9]:
min_setups = ["nominal", "g1z_pos_1em08", "ka_pos_1em08", "la_pos_1em08"] + ["mW_pos_1em08"]

In [10]:
# define nominal beam lvecs
analysis.Define("nominal_beam_e_lvec", "ROOT::Math::PxPyPzMVector(+8.750143e-01, 0., +1.250000e+02, +5.109968e-04)")
analysis.Define("nominal_beam_p_lvec", "ROOT::Math::PxPyPzMVector(+8.750143e-01, 0., -1.250000e+02, +5.109968e-04)")

In [11]:
analysis.book_sqme(
                      [
                          "nominal_beam_e_lvec",
                          "nominal_beam_p_lvec",
                          "true_lep_lvec",
                          "nomb_nu_lvec",
                          "true_quark1_lvec",
                          "true_quark2_lvec"
                      ],
                      "true_lep_charge",
                      "nurec_nomb_mc_cc10",
                      alt_setups=min_setups,
                      categories=signal_category,
                      hels=True
                     )
analysis.book_sqme(
                      [
                          "nominal_beam_e_lvec",
                          "nominal_beam_p_lvec",
                          "true_lep_lvec",
                          "nomb_nu_lvec",
                          "true_quark2_lvec",
                          "true_quark1_lvec",
                      ],
                      "true_lep_charge",
                      "wj_nurec_nomb_mc_cc10",
                      alt_setups=min_setups,
                      categories=signal_category,
                      hels=True
                     )

ValueError: Could not find "Define<::OmegaWrapperHelsFunctor>" (set cppyy.set_debug() for C++ errors):
  nullptr result where temporary expected

IncrementalExecutor::executeFunction: symbol '__opr_ww_i1_computation_MOD_new_event' unresolved while linking symbol '__cf_37'!
IncrementalExecutor::executeFunction: symbol '__opr_ww_i1_computation_MOD_init' unresolved while linking symbol '__cf_37'!
IncrementalExecutor::executeFunction: symbol '__opr_ww_i1_computation_MOD_color_sum' unresolved while linking symbol '__cf_37'!


In [ ]:
analysis.book_sqme(
                      [
                          "nominal_beam_e_lvec",
                          "nominal_beam_p_lvec",
                          "iso_lep_brems_lvec",
                          "clean_brems_nu_lvec",
                          "clean_brems_R2Jet_sel1_mlvec",
                          "clean_brems_R2Jet_sel2_mlvec",
                      ],
                      "iso_lep_charge",
                      "mlvec_clean_brems_reco_cc10",
                      alt_setups=min_setups,
                    #   categories=signal_category,
                      hels=True
                     )
analysis.book_sqme(
                      [
                          "nominal_beam_e_lvec",
                          "nominal_beam_p_lvec",
                          "iso_lep_brems_lvec",
                          "clean_brems_nu_lvec",
                          "clean_brems_R2Jet_sel2_mlvec",
                          "clean_brems_R2Jet_sel1_mlvec",
                      ],
                      "iso_lep_charge",
                      "wj_mlvec_clean_brems_reco_cc10",
                      alt_setups=min_setups,
                    #   categories=signal_category,
                      hels=True
                     )

In [ ]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec, write_categories=signal_category)

Info in <[ROOT.RDF] Info /tmp/root/spack-stage/spack-stage-root-6.38.00-2jf5cmbvudyjye7uzxzvsgmmlw2msfso/spack-build-2jf5cmb/include/ROOT/RDF/RInterface.hxx:1363 in auto ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager, void>::Snapshot(std::string_view, std::string_view, const ColumnNames_t &, const RSnapshotOptions &)::(anonymous class)::operator()() const [Proxied = ROOT::Detail::RDF::RLoopManager, DataSource = void]>: 
	In ROOT 6.38, the default compression settings of Snapshot have been changed from 101 (ZLIB with compression level 1, the TTree default) to 505 (ZSTD with compression level 5). This change may result in smaller Snapshot output dataset size by default. In order to suppress this message, set 'ROOT_RDF_SNAPSHOT_INFO=0' in your environment or set 'ROOT.RDF.Snapshot.Info: 0' in your .rootrc file.


In [ ]:
analysis.book_reports()

In [ ]:
%%time
analysis.run()

CPU times: user 3h 49min 3s, sys: 3min 30s, total: 3h 52min 34s
Wall time: 39min 56s


In [ ]:
# if write_outputs:
    # analysis.check_snapshots("events", output_path, checked_output_meta)

In [ ]:
analysis.print_reports()

         4f_sw_sl_signal               4f_sl_bkg
               0 (0e+00)               0 (0e+00) All
                   0e+00                   0e+00 efficiency

